## Get the Data

Either use the provided .csv file or (optionally) get fresh (the freshest?) data from running an SQL query on StackExchange: 

Follow this link to run the query from [StackExchange](https://data.stackexchange.com/stackoverflow/query/675441/popular-programming-languages-per-over-time-eversql-com) to get your own .csv file

<code>
select dateadd(month, datediff(month, 0, q.CreationDate), 0) m, TagName, count(*)
from PostTags pt
join Posts q on q.Id=pt.PostId
join Tags t on t.Id=pt.TagId
where TagName in ('java','c','c++','python','c#','javascript','assembly','php','perl','ruby','visual basic','swift','r','object-c','scratch','go','swift','delphi')
and q.CreationDate < dateadd(month, datediff(month, 0, getdate()), 0)
group by dateadd(month, datediff(month, 0, q.CreationDate), 0), TagName
order by dateadd(month, datediff(month, 0, q.CreationDate), 0)
</code>

## Import Statements

In [21]:
import pandas as pd

## Data Exploration

**Challenge**: Read the .csv file and store it in a Pandas dataframe

In [22]:
# read the data
data = pd.read_csv("/Users/acloudnerd/Desktop/acloudnerd/data-science/data-visualisation-basics/data/prog_lang_data.csv")

# rename the columns
data = data.rename(columns={'m' : 'DATE', 'TagName' : 'TAG', 'Unnamed: 2' : 'POSTS'})

**Challenge**: Examine the first 5 rows and the last 5 rows of the of the dataframe

In [23]:
df = pd.DataFrame(data)
df.head()

,DATE,TAG,POSTS
0,2008-07-01 00:00:00,c#,3
1,2008-08-01 00:00:00,assembly,8
2,2008-08-01 00:00:00,c,81
3,2008-08-01 00:00:00,c#,503
4,2008-08-01 00:00:00,c++,164


**Challenge:** Check how many rows and how many columns there are. 
What are the dimensions of the dataframe?

In [25]:
# number of (rows, columns)
df.shape

(3013, 3)

**Challenge**: Count the number of entries in each column of the dataframe

In [24]:
date_entries_count = df['DATE'].count()
tag_entries_count = df['TAG'].count()
post_entries_count = df['POSTS'].count()

print(f"We have {date_entries_count} date entries, {tag_entries_count} tag entries, and {post_entries_count} post entries.")

We have 3013 date entries, 3013 tag entries, and 3013 post entries.


**Challenge**: Calculate the total number of post per language.
Which Programming language has had the highest total number of posts of all time?

In [33]:
total_posts_per_language = df.groupby('TAG')['POSTS'].sum().sort_values(ascending=False)
total_posts_per_language

TAG
javascript    2521705
python        2204505
java          1914382
c#            1621715
php           1460900
c++            813981
r              510337
c              407965
swift          336108
ruby           229120
go              74457
perl            68295
delphi          52544
assembly        45073
Name: POSTS, dtype: int64

Some languages are older (e.g., C) and other languages are newer (e.g., Swift). The dataset starts in September 2008.

**Challenge**: How many months of data exist per language? Which language had the fewest months with an entry? 


In [52]:
months_of_data_per_language = df.groupby('TAG').size().reset_index(name='size')
min_row = months_of_data_per_language.loc[months_of_data_per_language['size'].idxmin()]
print(min_row)
months_of_data_per_language

TAG      go
size    202
Name: 5, dtype: object


,TAG,size
0,assembly,217
1,c,217
2,c#,218
3,c++,217
4,delphi,217
5,go,202
6,java,217
7,javascript,217
8,perl,217
9,php,217


## Data Cleaning

Let's fix the date format to make it more readable. We need to use Pandas to change format from a string of "2008-07-01 00:00:00" to a datetime object with the format of "2008-07-01"

## Data Manipulation



**Challenge**: What are the dimensions of our new dataframe? How many rows and columns does it have? Print out the column names and print out the first 5 rows of the dataframe.

**Challenge**: Count the number of entries per programming language. Why might the number of entries be different? 

## Data Visualisaton with with Matplotlib


**Challenge**: Use the [matplotlib documentation](https://matplotlib.org/3.2.1/api/_as_gen/matplotlib.pyplot.plot.html#matplotlib.pyplot.plot) to plot a single programming language (e.g., java) on a chart.

**Challenge**: Show two line (e.g. for Java and Python) on the same chart.

# Smoothing out Time Series Data

Time series data can be quite noisy, with a lot of up and down spikes. To better see a trend we can plot an average of, say 6 or 12 observations. This is called the rolling mean. We calculate the average in a window of time and move it forward by one overservation. Pandas has two handy methods already built in to work this out: [rolling()](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.rolling.html) and [mean()](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.core.window.rolling.Rolling.mean.html). 